In [164]:
from pulp import *
import pandas as pd

masses = pd.read_csv("../masses_all.tsv", sep="\t").set_index("nucleoside", drop=False)

In [165]:
masses

,nucleoside,monoisotopic_mass
nucleoside,,
A,A,267.09675
G,G,283.09167
C,C,243.08552
U,U,244.06954
1A,1A,281.11240
...,...,...
10G,10G,409.15970
102G,102G,425.15470
105G,105G,538.20230


In [186]:
import random
from scipy.stats import norm

nucleoside_re = re.compile(r"\d*[ACGU]")

true_sequence = nucleoside_re.findall("CG100GUU")
print(true_sequence)
n_fragments = 10

# sample random fragments from true sequence
fragment_noise = norm.rvs(scale=0.5, size=n_fragments)
def random_fragment():
    l = 0
    r = random.randint(l + 1, len(true_sequence))
    return l, r
fragments = [random_fragment() for _ in range(n_fragments)]
fragment_masses = [max(sum(masses.loc[b, "monoisotopic_mass"] for b in true_sequence[l:r]) + noise, 0.0) for noise, (l, r) in zip(fragment_noise, fragments)]
fragments, fragment_masses, fragment_noise

['C', 'G', '100G', 'U', 'U']


([(0, 5),
  (0, 2),
  (0, 1),
  (0, 2),
  (0, 5),
  (0, 2),
  (0, 2),
  (0, 4),
  (0, 5),
  (0, 2)],
 [1321.4993520634703,
  526.0893754368122,
  243.21661179017624,
  526.8054216716481,
  1321.3395594873105,
  526.47511816291,
  526.2604426051635,
  1077.3930814506177,
  1321.5120226156298,
  525.8534279666279],
 array([ 0.09138206, -0.08781456,  0.13109179,  0.62823167, -0.06841051,
         0.29792816,  0.08325261,  0.05465145,  0.10405262, -0.32376203]))

In [187]:
from itertools import combinations_with_replacement
from collections import namedtuple
from itertools import groupby
from dataclasses import dataclass, field
from typing import List

min_mass_diff = 1

n = len(fragment_masses)


Candidate = namedtuple("Candidate", ["seq", "mass", "abs_diff"])

@dataclass
class SeqEst:
    i: int
    prefix_mass: float
    sorted_masses: List[float]
    max_diff: float
    candidates: List = field(default_factory=list)

    def estimate(self):
        if self.i == len(self.sorted_masses):
            yield "", 0
        else:
            mass = self.sorted_masses[self.i]
            obs_suffix_mass = mass - self.prefix_mass
            candidates = []

            # case 1: single nucleotide
            for base, abs_diff in (masses["monoisotopic_mass"] - obs_suffix_mass).abs().items():
                mass = masses.loc[base, "monoisotopic_mass"]
                candidates.append(Candidate(base, mass, abs_diff))
            # case 2: two nucleotides
            for subseq in combinations_with_replacement(masses.index, 2):
                suffix_mass = masses.loc[list(subseq), "monoisotopic_mass"].sum()
                abs_diff = abs(suffix_mass - obs_suffix_mass)
                candidates.append(Candidate(f"[{''.join(subseq)}]", suffix_mass, abs_diff))
            # case 3: zero nucleotides
            candidates.append(Candidate("", 0, abs(obs_suffix_mass)))

            best_candidates = sorted(candidates, key=lambda candidate: candidate.abs_diff)
            last_diff = None
            for suffix_mass, same_mass_candidates in groupby(best_candidates, key=lambda candidate: candidate.mass):
                same_mass_candidates = list(same_mass_candidates)
                abs_diff = same_mass_candidates[0].abs_diff
                if last_diff is not None and abs_diff - last_diff > min_mass_diff:
                    break

                for next_seq, next_diff in self.recurse(self.i + 1, self.prefix_mass + suffix_mass):
                    for candidate in same_mass_candidates:
                        self.register_result(candidate.seq + next_seq, abs_diff + next_diff)
                last_diff = abs_diff

            # case 4: more nucleotides
            # for now: do not consider
            #print(self.i, obs_suffix_mass, self.prefix_mass, self.candidates, list(self.yield_results()))
            
            yield from self.yield_results()

    def recurse(self, i, mass):
        yield from SeqEst(i, mass, self.sorted_masses, self.max_diff).estimate()

    def register_result(self, seq, diff):
        self.candidates.append((seq, diff))

    def yield_results(self):
        yield from sorted(self.candidates, key=lambda item: item[1])[:2]

def estimate_sequence(fragment_masses):
    sorted_masses = sorted(fragment_masses)
    sorted_masses = sorted_masses[:n]
    max_diff = masses["monoisotopic_mass"].min() / 2.0
    estimates = sorted(SeqEst(0, 0, sorted_masses, max_diff).estimate(), key=lambda est: est[1])[:10]
    if not estimates:
        raise ValueError("Unable to determine reasonable estimates.")
    return estimates


In [147]:
def get_diff(obs_fragment_mass, seq):
    return abs(obs_fragment_mass - masses.loc[list(seq), "monoisotopic_mass"].sum())

In [167]:
sorted_idx = sorted(range(len(fragments)), key=lambda item: fragment_masses[item])
sorted_truth = pd.Series(fragments)[sorted_idx]
sorted_fragment_masses = pd.Series(fragment_masses)[sorted_idx]
sorted_truth, sorted_fragment_masses

(9     (0, 1)
 22    (0, 1)
 39    (0, 1)
 31    (0, 1)
 40    (0, 2)
 28    (0, 2)
 11    (0, 2)
 49    (0, 2)
 17    (0, 2)
 14    (0, 2)
 3     (0, 2)
 4     (0, 3)
 33    (0, 3)
 45    (0, 3)
 29    (0, 3)
 30    (0, 3)
 18    (0, 3)
 27    (0, 3)
 35    (0, 3)
 6     (0, 3)
 7     (0, 3)
 47    (0, 3)
 10    (0, 4)
 24    (0, 4)
 32    (0, 4)
 16    (0, 5)
 25    (0, 5)
 26    (0, 5)
 41    (0, 5)
 36    (0, 5)
 12    (0, 5)
 15    (0, 6)
 5     (0, 6)
 38    (0, 6)
 34    (0, 6)
 37    (0, 6)
 0     (0, 7)
 19    (0, 7)
 21    (0, 7)
 2     (0, 7)
 48    (0, 7)
 8     (0, 7)
 20    (0, 7)
 42    (0, 8)
 46    (0, 8)
 43    (0, 8)
 1     (0, 8)
 23    (0, 8)
 44    (0, 8)
 13    (0, 9)
 dtype: object,
 9      242.632650
 22     242.698529
 39     242.714774
 31     243.126011
 40     525.674676
 28     525.961909
 11     526.072353
 49     526.169117
 17     526.194582
 14     526.491784
 3      526.770374
 4      832.799298
 33     832.842138
 45     832.863195
 29     833.172895

In [188]:
estimate_sequence(fragment_masses)

[('CG[U100G]U', 1.8705774688650365), ('U19A[U100G]U', 2.6261638885125933)]

In [142]:
sum([get_diff(sorted_fragment_masses.loc[12], true_sequence[:12]) for mass in sorted_fragment_masses[4:5]])

0.12672880071067993

In [150]:
true_sequence

'CGGAUCGAUCUG'

In [60]:
list(product(masses.index, masses.index))

[('A', 'A'),
 ('A', 'G'),
 ('A', 'C'),
 ('A', 'U'),
 ('G', 'A'),
 ('G', 'G'),
 ('G', 'C'),
 ('G', 'U'),
 ('C', 'A'),
 ('C', 'G'),
 ('C', 'C'),
 ('C', 'U'),
 ('U', 'A'),
 ('U', 'G'),
 ('U', 'C'),
 ('U', 'U')]